## Learning repartition and coalesce 
> **分布式计算追求的高并发，本质上是靠“分区（Partitions）”来实现的。100个分区意味着100个小兵在并跑，这很好。但当计算接近尾声，数据被 `.filter()` 过滤得只剩几兆甚至几KB时，原本的100个分区就会在落盘（Write）时变成本地磁盘里的 100 个微型小文件。**
> **在分布式文件系统（如 HDFS、Cloud Storage 或 Delta Lake）里，【海量的小文件是系统性能的头号杀手】。元数据节点会被撑爆，下游任务读取时会因为频繁建立网络连接而彻底瘫痪。**
> **`repartition` 和 `coalesce` 就是为了调控全网分区数、在最后关头收拢小文件、或者是重新打散计算压力而被设计出来的“时空调度阀门”。它们决定了最终落盘的物理文件形态。**

---

## 一、 repartition 与 coalesce 的底层物理区别

虽然它们都能改变分区数，但底层的**网络代价和时空运行逻辑**有着天壤之折：

### 1. `repartition(N)`：全网大洗牌（强宽依赖）

* **物理机制**：不管你是要把分区调大还是调小，它都会**无条件触发一次全网的 Shuffle 宽依赖**。它会根据 Hash 规则把数据在内存和网络中彻底打碎，重新均匀分配到 $N$ 个新分区里。
* **时空画面**：数据在网络里盲目大搬家，硬件开销极高。

### 2. `coalesce(N)`：局部合并（窄依赖优化）

* **物理机制**：**只能用于减少分区数（$N < \text{当前分区数}$）**。它在底层非常聪明，**绝对不触发 Shuffle**。它是让同一个计算节点（Executor）上的几个相邻分区直接在内存里“原地合并”，或者让数据直接流向隔壁现成的分区。
* **时空画面**：数据在原地无痛靠拢，网线一丁点烟都不会冒，开销极低。
* ⚠️ **致命死角**：如果你强行用 `coalesce` 试图把分区调大（比如 `coalesce(200)` 原本只有100个），Spark 会假装没看见，**直接忽略不执行**。

---

## 二、 它们在重工业界的 3 大使用场景

### 场景 1：计算尾声，无痛收拢小文件 ➔ 必选 `coalesce`

* **业务特征**：原始数据 100 GB（200个分区），经过一层极其严苛的 `.filter()` 过滤后，全网只剩 10 MB。
* **优化策略**：在 `.write` 之前，反手套上一个 `.coalesce(1)`。
* **原理解析**：让 200 个空壳或极小分区在本地内存原地合一。最后落盘时，只会产生 1 个干净利落的大文件，**完美抹除 200 个小文件的灾难开销**。

### 场景 2：下游计算严重倾斜，强行重组并发 ➔ 必选 `repartition`

* **业务特征**：在进行复杂的机器学习或耗时的 CPU 死算前，发现数据全部堆在 2 个分区里，其余 98 个小兵没事干。
* **优化策略**：在耗时计算前，强行调用 `.repartition(100)`。
* **原理解析**：虽然付出了 Shuffle 的代价，但它把数据极其均匀地铺满了 100 个分区，让 100 个 CPU 核心同时轰鸣，**用网络开销换取了极限的计算并发度**。

### 场景 3：按业务字段进行物理隔离 ➔ 必选 `repartition("field")`

* **业务特征**：落盘时，希望相同国家、或相同日期的数据物理上存在同一个文件里。
* **优化策略**：`df.repartition("country").write...`
* **原理解析**：强行按该字段进行 Hash 洗牌，确保相同特征的数据完美归拢。


##### Task: 制造小文件风暴


In [0]:
df_small_storm = spark.range(0,1000)

# 写入临时delta表
df_small_storm.write.format("delta").mode("overwrite").saveAsTable("yuto_file_storm")

In [0]:
# 查询一下底层生成了多少个物理文件

detail_df = spark.sql("DESCRIBE DETAIL yuto_file_storm")
files_count = detail_df.select("numFiles").collect()[0][0]

In [0]:
print(f"❌ 未经优化的写入：总共生成了 {files_count} 个碎片小文件！")

##### Experiment B : Using coalesce(_the number of file that you wanna ooptimize into_)

In [0]:
# 1. 同样是 1000 条数据
df_clean = spark.range(0, 1000)

# 2. 🚀 架构师神级阀门：在落盘前最后一秒钟，强行无痛收拢为 1 个分区
df_optimized = df_clean.coalesce(1)

# 3. 🎉 安全落盘：写入属于你的另一个 Delta 托管表
df_optimized.write.format("delta").mode("overwrite").saveAsTable("yuto_file_storm_clean")

# 4. 再次对账文件数量
optimized_detail_df = spark.sql("DESCRIBE DETAIL yuto_file_storm_clean")
optimized_files_count = optimized_detail_df.select("numFiles").collect()[0][0]

print(f"🎉 经过 coalesce(1) 优化：总共生成了 {optimized_files_count} 个完美的物理大文件！")

%md
### 大数据重工业底座：repartition 与 coalesce 控制分区数避坑指南

#### 1. 核心物理对账表

| 算子维度 | `repartition(N)` | `coalesce(N)` |
| :--- | :--- | :--- |
| **Shuffle 成本** |  **高**（无条件触发全网大洗牌） |  **零**（纯内存本地合并） |
| **分区数调整** |  可变大，可变小 | ⬇ **只能变小**（强行变大直接被忽略） |
| **数据均匀度** |  极均匀（基于 Hash 算法重组） |  可能不均匀（取决于原本的分区块大小） |

#### 2. 工业界避坑红线
* **【落盘前防线】**：在 `.write` 之前，如果发现过滤后的数据量极小，**死死卡住，必须加上 `.coalesce(N)`**。严禁将数百个空壳分区直接倒进存储，引发小文件灾难。
* **【调大分区必选】**：如果因为下游计算太重、想把分区从 10 提升到 200 释放并发度，**必须用 `.repartition(200)`**，此时用 `coalesce` 毫无效果。
